# Stage 3 — Finance DPO Preference Alignment
This standalone notebook creates an SFT warm start and then performs DPO preference alignment on Finance FAQ responses. It includes SFT-vs-DPO evaluation, adapter saving, and a Finance-only ipywidgets demo. The SFT warm start is included because DPO should begin from an instruction-tuned model.

In [1]:
# Install pinned, Colab-compatible packages
!pip install -q "unsloth[colab-new]" "trl>=0.18,<0.26" "transformers>=4.51,<5" datasets accelerate bitsandbytes ipywidgets pandas matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from pathlib import Path
import json, warnings, torch
from datasets import Dataset
warnings.filterwarnings("ignore")

DOMAIN = "Finance FAQ Assistant"
DOMAIN_OPTIONS = [DOMAIN]
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True
SEED = 42

print("Domain:", DOMAIN)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


Domain: Finance FAQ Assistant
GPU: Tesla T4


In [3]:
# Locate extracted data directory (no ZIP upload required).
candidates = [
    Path("../data"),                # local run from notebooks/
    Path("./data"),                 # local run from project root
    Path("/content/data"),          # Colab with uploaded/extracted data folder
    Path("/data"),                  # absolute data mount
    Path("/content/finance_stage1_data"),  # backward compatibility
    Path("/content/finance_stage2_data"),
    Path("/content/finance_stage3_data"),
]

DATA_DIR = next((p for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find extracted data folder. Expected one of: " + ", ".join(str(p) for p in candidates)
    )

DATA_DIR = DATA_DIR.resolve()
print("Using DATA_DIR:", DATA_DIR)
print("Available files:", sorted(p.name for p in DATA_DIR.glob("*") if p.is_file()))


Using DATA_DIR: /content/data
Available files: ['preference_dataset.jsonl']


In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Model and QLoRA adapter initialized.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.7.2 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Model and QLoRA adapter initialized.


In [5]:
# Define shared Step 10 question set and base-model outputs before any stage-3 training.
EVAL_QUESTIONS = [
    "What is a savings account?",
    "What is the difference between savings and current account?",
    "What documents are required for KYC?",
    "How do I transfer money using NEFT?",
    "What is RTGS and when should I use it?",
    "What is a credit score and why does it matter?",
    "How can I apply for a personal loan?",
    "What should I do if I receive a suspicious OTP request?",
    "My EMI payment failed. What should I do?",
    "How can I improve my credit score?",
]

def ask_model(question, max_new_tokens=240, temperature=0.2):
    prompt = f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{question}\n\n### Assistant:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 0.01),
        top_p=0.9,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
base_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
for q, a in base_answers.items():
    print("\nQ:", q, "\nBase:", a)


Q: What is a savings account? 
Base: A savings account is a type of bank account that offers you the opportunity to save money for future use, such as buying a car or paying off debt. Savings accounts typically offer higher interest rates than checking accounts and can be used to invest in stocks, bonds, and other financial instruments. They are also often offered with lower fees compared to checking accounts. It's important to note that it's always best to consult with a financial advisor before opening any new accounts. 

Savings accounts are available at banks, credit unions, and online banking platforms. You can open an account by visiting your local branch, calling 211, or using the online banking platform. The account will have a unique account number, which you can share with friends and family members who want to open an account on your behalf. 

Remember, it's important to understand the risks associated with saving accounts, including the possibility of losing money if you w

## Build SFT warm-start dataset

In [7]:
instruction_rows=[json.loads(x) for x in (DATA_DIR/"instruction_dataset.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
def sft_text(r):
    return {"text":f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['instruction'].strip()}\n\n### Assistant:\n{r['response'].strip()}{tokenizer.eos_token}"}
sft_dataset=Dataset.from_list([sft_text(r) for r in instruction_rows])
print("SFT examples:",len(sft_dataset))

SFT examples: 105


In [8]:
from trl import SFTTrainer, SFTConfig
SFT_DIR=Path("/content/finance_adapters/stage3_sft_warm_start")
sft_args=SFTConfig(output_dir=str(SFT_DIR),dataset_text_field="text",max_length=MAX_SEQ_LENGTH,per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=2e-4,logging_steps=10,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
sft_trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=sft_dataset,args=sft_args)
sft_trainer.train(); model.save_pretrained(str(SFT_DIR)); tokenizer.save_pretrained(str(SFT_DIR))
print("Warm-start adapter saved:",SFT_DIR)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/105 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 105 | Num Epochs = 1 | Total steps = 27
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
10,2.418700
20,1.608900


Warm-start adapter saved: /content/finance_adapters/stage3_sft_warm_start


## SFT evaluation snapshot

In [9]:
FastLanguageModel.for_inference(model)
sft_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
for q, a in sft_answers.items():
    print("\nQ:", q, "\nSFT:", a)


Q: What is a savings account? 
SFT: A savings account offers convenient deposit and withdrawal services, lower interest rates, and no annual fees. It's ideal for short-term investments or emergency funds.

Q: What is the difference between savings and current account? 
SFT: Savings accounts offer higher interest rates on deposits than current accounts. Savings can be accessed with any bank branch, while current accounts require a passbook or online banking to access funds. Current accounts have lower interest rates but no overdraft facilities.

Q: What documents are required for KYC? 
SFT: KYC includes identification documents such as Aadhaar card, PAN card, passport, and birth certificate. You can also provide a PAN card if you have an existing bank account or a driving license if you are a driver.

Q: How do I transfer money using NEFT? 
SFT: NEFT (National Electronic Fund Transfer) is a secure and fast method of transferring funds between banks. You can initiate a NEFT transaction 

## Build DPO dataset

In [10]:
preference_rows=[json.loads(x) for x in (DATA_DIR/"preference_dataset.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
def dpo_row(r):
    prompt=f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['prompt'].strip()}\n\n### Assistant:\n"
    return {"prompt":prompt,"chosen":r['chosen'].strip()+tokenizer.eos_token,"rejected":r['rejected'].strip()+tokenizer.eos_token}
dpo_dataset=Dataset.from_list([dpo_row(r) for r in preference_rows])
print("Preference examples:",len(dpo_dataset)); print(dpo_dataset[0])

Preference examples: 51
{'prompt': '### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\nHow can I apply for a personal loan?\n\n### Assistant:\n', 'chosen': "You can apply for a personal loan online through the bank's website or mobile app, or by visiting a branch. You will need to submit your identity proof, address proof, salary slips, income tax returns, and bank statements for the last 3-6 months. The bank will assess your credit score, income, and repayment capacity before approving the loan.<|im_end|>", 'rejected': 'Just go to the bank and ask for money.<|im_end|>'}


## Train DPO alignment

In [11]:
from trl import DPOTrainer, DPOConfig
model.train()
DPO_DIR=Path("/content/finance_adapters/stage3_dpo")
dpo_args=DPOConfig(output_dir=str(DPO_DIR),per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=5e-6,beta=0.1,max_length=MAX_SEQ_LENGTH,max_prompt_length=512,logging_steps=5,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
dpo_trainer=DPOTrainer(model=model,ref_model=None,args=dpo_args,train_dataset=dpo_dataset,processing_class=tokenizer)
dpo_trainer.train(); model.save_pretrained(str(DPO_DIR)); tokenizer.save_pretrained(str(DPO_DIR))
print("DPO adapter saved:",DPO_DIR)

Extracting prompt in train dataset (num_proc=5):   0%|          | 0/51 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=5):   0%|          | 0/51 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=5):   0%|          | 0/51 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51 | Num Epochs = 1 | Total steps = 13
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.155800,3.637775,1.441322,1.000000,2.196454,-170.983032,-48.697765,-1.703613,-1.415920
10,0.087200,3.854259,1.155705,1.000000,2.698555,-174.596741,-48.571175,-1.558399,-1.472526


DPO adapter saved: /content/finance_adapters/stage3_dpo


## Step 10 - Final evaluation (Base vs SFT vs DPO)

In [12]:
import re
import pandas as pd
from pathlib import Path
from IPython.display import display

def resolve_reports_dir():
    candidates = []
    if "DATA_DIR" in globals():
        candidates.append(Path(DATA_DIR).resolve().parent / "reports")
    candidates.extend([Path.cwd() / "reports", Path("/content/reports"), Path("./reports"), Path("../reports")])

    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            probe = p / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return p.resolve()
        except Exception:
            continue
    raise RuntimeError("Could not create a writable reports directory.")

def score_answer(question, answer):
    q = question.lower()
    a = answer.lower().strip()
    words = re.findall(r"[a-zA-Z]+", a)
    score = 0
    notes = []

    if len(words) >= 30:
        score += 2
    elif len(words) >= 18:
        score += 1
    else:
        notes.append("too brief")

    domain_terms = [
        "kyc", "neft", "rtgs", "imps", "upi", "emi", "cibil",
        "interest", "loan", "deposit", "bank", "account", "ifsc", "otp"
    ]
    domain_hits = sum(1 for t in domain_terms if t in a)
    if domain_hits >= 2:
        score += 2
    elif domain_hits == 1:
        score += 1
    else:
        notes.append("low domain specificity")

    if "otp" in q or "unauthorized" in q or "suspicious" in q:
        if any(k in a for k in ["never share", "fraud", "report", "block", "cybercrime", "helpline"]):
            score += 2
        else:
            notes.append("missing safety guidance")

    if any(k in q for k in ["how do i", "how can i", "what should i do"]):
        if any(k in a for k in ["log in", "visit", "submit", "documents", "steps", "contact", "immediately"]):
            score += 1
        else:
            notes.append("weak actionable steps")

    if any(k in a for k in ["it depends", "usually", "generally", "contact your bank"]):
        score -= 1
        notes.append("generic phrasing")

    return score, notes

def pick_best(question, base_ans, sft_ans, dpo_ans):
    b_score, b_notes = score_answer(question, base_ans)
    s_score, s_notes = score_answer(question, sft_ans)
    d_score, d_notes = score_answer(question, dpo_ans)

    ranked = [
        ("Base Model", b_score, b_notes),
        ("SFT Model", s_score, s_notes),
        ("DPO Model", d_score, d_notes),
    ]
    ranked.sort(key=lambda x: x[1], reverse=True)

    top_name, top_score, top_notes = ranked[0]
    second_name, second_score, _ = ranked[1]

    if top_score == second_score:
        best = "Tie"
        reason = f"Top responses are similar in quality (score {top_score})."
    else:
        best = top_name
        reason = f"{top_name} is more complete/specific (score {top_score} vs {second_score})."

    note_text = ", ".join(top_notes[:2]) if top_notes else "no major weakness detected"
    reason += f" Diagnostic notes: {note_text}."
    return best, reason, b_score, s_score, d_score

FastLanguageModel.for_inference(model)
dpo_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}

rows = []
best_counts = {"Base Model": 0, "SFT Model": 0, "DPO Model": 0, "Tie": 0}
dpo_minus_sft = []

for q in EVAL_QUESTIONS:
    best, reason, b_score, s_score, d_score = pick_best(q, base_answers[q], sft_answers[q], dpo_answers[q])
    best_counts[best] += 1
    dpo_minus_sft.append(d_score - s_score)
    rows.append({
        "Question": q,
        "Base Model Answer": base_answers[q],
        "SFT Model Answer": sft_answers[q],
        "DPO Model Answer": dpo_answers[q],
        "Best Answer": best,
        "Reason": reason,
    })

df_final = pd.DataFrame(rows)
display(df_final)

# Export Step 10 table to reports/final_evaluation.md
reports_dir = resolve_reports_dir()
out_path = reports_dir / "final_evaluation.md"

lines = [
    "# Final Evaluation (Base vs SFT vs DPO)",
    "",
    "This report compares all three stages of model quality:",
    "",
    "1. Base model",
    "2. Instruction fine-tuned (SFT) model",
    "3. DPO-aligned model",
    "",
    "## Evaluation Criteria",
    "",
    "- Correctness",
    "- Helpfulness",
    "- Domain accuracy",
    "- Safety",
    "- Tone",
    "- Clarity",
    "- Hallucination reduction",
    "- Professional response quality",
    "",
    "## Final Comparison Table",
    "",
    "| Question | Base Model Answer | SFT Model Answer | DPO Model Answer | Best Answer | Reason |",
    "| --- | --- | --- | --- | --- | --- |",
]
for row in rows:
    q = " ".join(row["Question"].split()).replace("|", "\\|")
    base = " ".join(row["Base Model Answer"].split()).replace("|", "\\|")
    sft = " ".join(row["SFT Model Answer"].split()).replace("|", "\\|")
    dpo = " ".join(row["DPO Model Answer"].split()).replace("|", "\\|")
    best = row["Best Answer"].replace("|", "\\|")
    reason = " ".join(row["Reason"].split()).replace("|", "\\|")
    lines.append(f"| {q} | {base} | {sft} | {dpo} | {best} | {reason} |")

avg_gain = sum(dpo_minus_sft) / len(dpo_minus_sft) if dpo_minus_sft else 0.0
lines += [
    "",
    "## Overall Findings",
    "",
    f"- Stage with largest quality jump (by wins): DPO={best_counts['DPO Model']}, SFT={best_counts['SFT Model']}, Base={best_counts['Base Model']}, Tie={best_counts['Tie']}",
    f"- Safety and professionalism signal (avg DPO-SFT score delta): {avg_gain:.2f}",
    "- Final model limitations: Some responses may remain concise or partially generic; manual review is recommended for high-stakes finance queries.",
]

out_path.write_text("\n".join(lines), encoding="utf-8")
print("Step 10 report written:", out_path)

,Question,Base Model Answer,SFT Model Answer,DPO Model Answer,Best Answer,Reason
0,What is a savings account?,A savings account is a type of bank account th...,A savings account offers convenient deposit an...,A savings account offers convenient deposit an...,Tie,Top responses are similar in quality (score 4)...
1,What is the difference between savings and cur...,Savings refers to an investment of money that ...,Savings accounts offer higher interest rates o...,Savings accounts offer higher interest rates o...,Tie,Top responses are similar in quality (score 4)...
2,What documents are required for KYC?,"To initiate the KYC process, you will need to ...",KYC includes identification documents such as ...,KYC includes identification documents such as ...,Tie,Top responses are similar in quality (score 4)...
3,How do I transfer money using NEFT?,NEFT (National Electronic Funds Transfer) is a...,NEFT (National Electronic Fund Transfer) is a ...,NEFT (National Electronic Funds Transfer) is a...,Base Model,Base Model is more complete/specific (score 5 ...
4,What is RTGS and when should I use it?,RTGS stands for Real-Time Gross Settlement. It...,RTGS (Real Time Gross Settlement) is a mobile ...,"RTGS (Real Time Gifting) is a fast, secure, an...",Tie,Top responses are similar in quality (score 4)...
5,What is a credit score and why does it matter?,A credit score is a numerical value assigned t...,A credit score is a numerical rating that bank...,A credit score is a numerical rating that lend...,Tie,Top responses are similar in quality (score 4)...
6,How can I apply for a personal loan?,Certainly! Here's how you can apply for a pers...,You can apply online or in person at any bank ...,You can apply online or in person at any bank ...,Base Model,Base Model is more complete/specific (score 5 ...
7,What should I do if I receive a suspicious OTP...,If you receive an OTP request from a financial...,Always verify the sender's identity and contac...,Always verify the sender's identity through ph...,Base Model,Base Model is more complete/specific (score 7 ...
8,My EMI payment failed. What should I do?,Certainly! Here's what you need to do:\n\n1. *...,"If your EMI payment fails, you can try the fol...","If your EMI payment fails, you can try the fol...",Tie,Top responses are similar in quality (score 4)...
9,How can I improve my credit score?,Improving your credit score is an important st...,Credit scores are based on your payment histor...,Credit scores are influenced by your payment h...,Base Model,Base Model is more complete/specific (score 5 ...


Step 10 report written: /content/reports/final_evaluation.md


## Interactive demo

In [14]:
import html, ipywidgets as widgets
from IPython.display import display, HTML

FastLanguageModel.for_inference(model)

domain_dd = widgets.Dropdown(options=DOMAIN_OPTIONS, value=DOMAIN, description="Domain", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
question_box = widgets.Textarea(value="What is the difference between NEFT and RTGS?", description="Question", style={"description_width":"100px"}, layout=widgets.Layout(width="900px", height="110px"))
max_tokens = widgets.IntSlider(value=220, min=80, max=400, step=20, description="Max tokens", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
temperature = widgets.FloatSlider(value=0.2, min=0.1, max=0.8, step=0.05, description="Temperature", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
button = widgets.Button(description="Generate Finance Answer", button_style="primary", layout=widgets.Layout(width="260px", height="44px"))
status = widgets.HTML()
answer = widgets.HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will appear here.</div>")

def render(text):
    return f"<div style='padding:22px;border:1px solid #bfdbfe;border-radius:16px;background:linear-gradient(180deg,#fff,#eff6ff);line-height:1.7; color: black;'><b style='color:#1e3a8a'>Finance Assistant Answer</b><hr>{html.escape(text).replace(chr(10),'<br>')}</div>"

def on_click(_):
    q=question_box.value.strip()
    if not q:
        answer.value=render("Please enter a finance-related question.")
        return
    button.disabled=True; status.value="<b style='color:#2563eb'>Generating...</b>"
    try:
        torch.cuda.empty_cache()
        answer.value=render(ask_model(q, max_new_tokens=max_tokens.value, temperature=temperature.value))
        status.value="<b style='color:#15803d'>Completed</b>"
    except Exception as exc:
        answer.value=render(f"{type(exc).__name__}: {exc}")
        status.value="<b style='color:#b91c1c'>Failed</b>"
    finally:
        button.disabled=False
button.on_click(on_click)

display(HTML("""<div style='background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:26px;border-radius:18px;text-align:center;margin-bottom:18px'><h1>Enterprise Finance FAQ Assistant</h1><p>Standalone fine-tuning stage demonstration</p></div>"""))
display(domain_dd, max_tokens, temperature, question_box, button, status, answer)


Dropdown(description='Domain', layout=Layout(width='580px'), options=('Finance FAQ Assistant',), style=Descrip…

IntSlider(value=220, description='Max tokens', layout=Layout(width='580px'), max=400, min=80, step=20, style=S…

FloatSlider(value=0.2, description='Temperature', layout=Layout(width='580px'), max=0.8, min=0.1, step=0.05, s…

Textarea(value='What is the difference between NEFT and RTGS?', description='Question', layout=Layout(height='…

Button(button_style='primary', description='Generate Finance Answer', layout=Layout(height='44px', width='260p…

HTML(value='')

HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will…